## <국립민속박물관 스크래핑>

### 0. 필요한 패키지 설치
- beautifulsoup, pandas, openpyxl

### 1. 접속 준비 (세션 만들기, CSRF 토큰 만들기)
- 국립민속박물관 검색 기능은 보안을 위해 **CSRF 토큰**이라는 임시 값을 요구
    - 이 요청이 실제로 저 검색 페이지를 열어본 사람이 보낸 게 맞다는 걸 증명하기 위함
- 먼저 검색 페이지에 평범하게 접속(GET)해서, 페이지 안에 숨어있는 CSRF 토큰 값 읽기
- 이때 받은 쿠키(로그인 상태 같은 걸 기억하는 값)와 토큰을 계속 재사용해서 이후 요청 보내기

In [1]:
# 1. 필요한 패키지, 라이브러리 불러오기
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# 2. URL
BASE = "https://www.nfm.go.kr"
SEARCH_LIST_URL = f"{BASE}/user/data/home/101/DataRelicCategoryList.do"
DETAIL_URL = f"{BASE}/user/data/home/101/DataRelicView.do"

# 실제 브라우저처럼 보이도록 User-Agent를 지정해줌 -> 일부 서버는 이게 없으면 차단하기 때문
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

# 3. 쿠키를 자동으로 기억해주는 과정 (쿠키 관리해올 필요 없게)
session = requests.Session()
session.headers.update(HEADERS)

# 4. CSRF 토큰 읽기 (검색해서 읽어오는 것)
def get_csrf_token():
    resp = session.get(f"{BASE}/user/data/home/101/DataRelicCategoryList.do")
    soup = BeautifulSoup(resp.text, "html.parser")
    token_input = soup.select_one('input[name="CSRFToken"]')
    if token_input is None:
        raise RuntimeError("CSRF 토큰을 찾지 못했습니다. 사이트 구조가 바뀌었을 수 있어요.")
    return token_input.get("value")

csrf_token = get_csrf_token()
print("CSRF 토큰:", csrf_token)

CSRF 토큰: 9ca2d97c-5295-4eb6-9bc3-57f99351183d


### 2. 검색어로 목록 긁어오기
- 검색 결과 목록은 한 번에 최대 몇십~백 개씩만 보여주기 때문에, 페이지를 넘겨가며 계속 요청해서 더 이상 결과가 없을 때까지 반복
- 각 소장품은 상세 페이지 주소에 **seq=긴영문숫자** 형태의 고유 번호를 가짐 ex) seq=PS0100200100100508000000
- 여기서는 이름표(제목)와 고유 번호만 먼저 수집
- 목록 페이지 자체에는 명칭 말고 다른 정보가 별로 없어서, 상세 페이지를 따로 열어야 모든 필드 조사 가능

In [2]:
# 데이터 수집 함수
# keyword: 키워드
# page_size: 한 번에 몇 건씩 가져올지
# delay: 페이지 요청 사이 쉬는 시간(너무 많으면 서버에 부담)
# max_pages: 무한 루프 방지
def search_relic_list(keyword, page_size=50, delay=0.5, max_pages=200):
    results = []
    page_no = 1

# page_no를 max_pages 만큼 반복
# cf) page_no를 인자로 두지 않는 이유는 애초에 page_no는 1부터 시작하는 게 디폴트이기 때문
    while page_no <= max_pages:
        payload = {
            "query": keyword,
            "pageNo": str(page_no),
            "pageRow": str(page_size),
            "CSRFToken": csrf_token,
            "collection": "coll_search",
            "searchField": "ALL",   # ALL = 명칭+설명 등 전체 범위에서 검색 (사이트 기본 검색과 동일)
        }
        resp = session.post(SEARCH_LIST_URL, data=payload)
        soup = BeautifulSoup(resp.text, "html.parser")

        # 상세페이지로 연결되는 링크들만 모두 찾음
        links = soup.select('a[href*="DataRelicView.do"]')

        if not links:
            # 더 이상 결과가 없으면(빈 페이지) 종료
            break

        for a in links:
            href = a.get("href", "")
            m = re.search(r"seq=([A-Za-z0-9]+)", href)
            if not m:
                continue
            seq = m.group(1)
            title = a.get_text(strip=True)
            results.append({"seq": seq, "list_title": title, "searched_keyword": keyword})

        print(f"[{keyword}] {page_no}페이지: {len(links)}건 수집 (누적 {len(results)}건)")
        page_no += 1
        time.sleep(delay)

    return results

### 3. 모든 페이지 긁어오기 (상세페이지)

- 상세 페이지(DataRelicView.do?seq=...) 구조
---
```html
<div class="d-relic__data">
    <div class="d-relic__key">소장품 명칭</div>
    <div class="d-relic__value">흉배(胸背)</div>
</div>
<div class="d-relic__data">
    <div class="d-relic__key">국적/시대</div>
    <div class="d-relic__value">한국-조선</div>
</div>
... (용도/기능, 크기, 소장품 번호, 내용 등이 같은 방식으로 반복됨)
```
---

- .d-relic__data가 **항목 하나(라벨+값)**
    - 품목 종류에 따라 라벨 개수가 다를 수 있어서(어떤 건 필드가 더 많거나 적을 수 있음), 라벨 이름을 코드에 미리 고정해두지 않고, 페이지에 있는 라벨을 그대로 열 이름(컬럼명)으로 사용
- '모든 필드'를 빠짐없이, 자동으로 가져올 수 있음

In [3]:
# 소장품 고유 변호로 상세 페이지에 접속해서 페이지에 있는 '라벨: 값' 쌍을 딕셔너리로 처리하는 함수
def get_relic_detail(seq, delay=0.5):
    url = f"{DETAIL_URL}?seq={seq}"
    resp = session.get(url)
    soup = BeautifulSoup(resp.text, "html.parser")

    data = {"seq": seq, "detail_url": url}

    # 화면에 보이는 모든 '라벨-값'
    for block in soup.select(".d-relic__data"):
        key_el = block.select_one(".d-relic__key")
        val_el = block.select_one(".d-relic__value")
    
        if key_el is None or val_el is None:
            continue
        key = key_el.get_text(strip=True)
        # 값 안에 줄바꿈/여러 칸 공백이 많으므로 한 줄로 깔끔하게 정리
        value = re.sub(r"\s+", " ", val_el.get_text(separator=" ", strip=True))
        data[key] = value

    # 대표 이미지 주소 찾기 (relic 썸네일 이미지 중 첫 번째)
    img = soup.select_one('img[src*="/common/apithumb/relic/"]')
    data["image_url"] = (BASE + img["src"]) if img and img["src"].startswith("/") else (img["src"] if img else None)

    time.sleep(delay)
    return data

### 4. 실행
- 검색어 목록(KEYWORDS) 기준으로 실행
- 목록(제목+번호)만 다 모으고, 번호 하나하나마다 상세 페이지 열기
- 같은 소장품이 여러 검색어에 동시에 걸리는 것을 방지하기 위해 seq 기준으로 중복을 제거

In [4]:
# 1. 키워드 기반 수집
KEYWORDS = ["단령"]

all_list_items = []
for kw in KEYWORDS:
    all_list_items.extend(search_relic_list(kw))

print("검색된 전체 건수(중복 포함):", len(all_list_items))

[단령] 1페이지: 50건 수집 (누적 50건)
[단령] 2페이지: 50건 수집 (누적 100건)
[단령] 3페이지: 26건 수집 (누적 126건)
검색된 전체 건수(중복 포함): 126


In [5]:
# 2. 데이터 프레임 제작
list_df = pd.DataFrame(all_list_items)
list_df = list_df.drop_duplicates(subset="seq").reset_index(drop=True)
print("중복 제거 후 소장품 수:", len(list_df))
list_df.head()

중복 제거 후 소장품 수: 126


,seq,list_title,searched_keyword
0,PS0100200100102806300000,단령(團領),단령
1,PS0100200100102806400000,단령(團領),단령
2,PS0100200100104555500000,단령(團領),단령
3,PS0100200100107759300000,단령(團領),단령
4,PS0100200100104555300000,단령(團領),단령


In [6]:
# 3. 상세 페이지 열기
details = []
for i, row in list_df.iterrows():
    detail = get_relic_detail(row["seq"])
    detail["searched_keyword"] = row["searched_keyword"]
    detail["list_title"] = row["list_title"]
    details.append(detail)
    if (i + 1) % 20 == 0:
        print(f"{i + 1} / {len(list_df)} 건 완료")

print("총", len(details), "건")

20 / 126 건 완료
40 / 126 건 완료
60 / 126 건 완료
80 / 126 건 완료
100 / 126 건 완료
120 / 126 건 완료
총 126 건


### 5. 표로 정리 + 파일 저장

In [12]:
# 1. 데이터프레임 확인
df = pd.DataFrame(details)

# 2. 컬럼 순서 정리
priority_cols = ["searched_keyword", "소장품 명칭", "list_title", "국적/시대", "용도/기능",
                 "크기", "소장품 번호", "내용", "image_url", "detail_url", "seq"]
other_cols = [c for c in df.columns if c not in priority_cols]
ordered_cols = [c for c in priority_cols if c in df.columns] + other_cols
df = df[ordered_cols]


dallyeong = df[df["소장품 명칭"].str.contains("단령", na=False)]
print(dallyeong.shape)
dallyeong.head()

(51, 11)


,searched_keyword,소장품 명칭,list_title,국적/시대,용도/기능,크기,소장품 번호,내용,image_url,detail_url,seq
0,단령,단령(團領),단령(團領),한국-조선,의-의류-의례복-남자수의,품 : 54 길이 : 126 화장 : 121,028063,둥근 깃의 남성용 겉옷. 충남 태안군 태안읍 삭선리 남오성(1643~1712년) 묘...,https://www.nfm.go.kr/common/apithumb/relic/83...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100102806300000
1,단령,단령(團領),단령(團領),한국-조선,의-의류-의례복-남자수의,화장 : 120.5 길이 : 125 품 : 57,028064,둥근 깃의 남성용 겉옷. 충남 태안군 태안읍 삭선리 남오성(1643~1712) 묘 ...,https://www.nfm.go.kr/common/apithumb/relic/85...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100102806400000
2,단령,단령(團領),단령(團領),한국-조선,사회생활-의례생활-상장 / 의-의류-평상복-남자포류,길이 : 131.5 품 : 55 화장 : 125.5,045555,"남성용 포(袍). 이진숭(李鎭嵩, 1702~1756) 묘에서 출토된 보공품(補空品)...",https://www.nfm.go.kr/common/apithumb/relic/83...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100104555500000
3,단령,단령(團領),단령(團領),한국-조선,사회생활-의례생활-상장 / 의-의류-의례복-남자수의,품 : 50 화장 : 118 길이 : 136,077593,"남성용 포(袍). 신광헌(申光憲, 1731~1784) 묘(경기도 남양주 별내면 화접...",https://www.nfm.go.kr/common/apithumb/relic/84...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100107759300000
4,단령,단령(團領),단령(團領),한국-조선,사회생활-의례생활-상장 / 의-의류-평상복-남자포류,품 : 52.5 길이 : 129 화장 : 125,045553,"남성용 포(袍). 이진숭(李鎭嵩, 1702~1756) 묘에서 출토된 보공품(補空品)...",https://www.nfm.go.kr/common/apithumb/relic/83...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100104555300000


In [14]:
# 3. 파일 저장(엑셀)
dallyeong.to_excel("../data/nfm_dallyeong.xlsx", index=False)

# 4. 이미지 저장
dallyeong = dallyeong.reset_index(drop=True) 

for i, row in dallyeong.iterrows():
    seq = row["seq"]
    image_url = row["image_url"]

    if pd.isna(image_url):
        continue

    filepath = f"../image/dallyeong/{seq}.jpg"

    resp = session.get(image_url)
    with open(filepath, "wb") as f:
        f.write(resp.content)

    if (i + 1) % 51 == 0:
        print(f"{i + 1} / {len(dallyeong)} 완료")

    time.sleep(0.5)

51 / 51 완료
